# Chapter 29 — Generative Adversarial Networks II: Conditional GANs, CycleGAN & StyleGAN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part4_transformers_genai/ch29_generative_adversarial_networks_ii/ch29_notebook.ipynb)

Companion notebook for **Chapter 29** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Conditioning both networks on extra information C is what gives you control over what is generated.
- In an ACGAN the discriminator does double duty: it judges realness AND predicts the class label.
- pix2pix-style translation needs paired images; CycleGAN removes that requirement using cycle consistency.
- Cycle consistency means translating an image across and back should return the original, which stops the generator ignoring its input.
- StyleGAN controls image style layer by layer using AdaIN, with added noise producing fine stochastic detail.

## Example 1: Conditional GAN Inputs and ACGAN Outputs

This shows exactly how the conditioning information C enters a conditional GAN: the class label is one-hot encoded and concatenated with the noise vector, so the same z with a different label produces a different image. The second half shows the ACGAN discriminator's two outputs - the probability the image is real and the predicted class label from the auxiliary classifier.

In [1]:
# Chapter 29 - Example 1: Conditional GAN inputs + ACGAN dual outputs
import numpy as np
np.random.seed(0)
np.set_printoptions(suppress=True)

print("=== Conditional GAN (cGAN) ===\n")
print("Both G and D are conditioned on additional information C.")
print("C can be a class label, a source-domain image, a sketch, or text.\n")

n_classes, z_dim = 5, 6
classes = ["cat", "dog", "bird", "car", "tree"]

def one_hot(k, n):
    v = np.zeros(n); v[k] = 1.0; return v

# ---- Class-conditional GAN ----
print("--- Class-Conditional GAN: C = class label ---\n")
z = np.round(np.random.randn(z_dim), 2)
label = 2                                    # want to generate a "bird"
c = one_hot(label, n_classes)
g_input = np.concatenate([z, c])

print(f"Random noise vector z ({z_dim}-dim)      : {z}")
print(f"Class label            : {label} = '{classes[label]}'")
print(f"One-hot encoding C     : {c}")
print(f"Generator input [z ; C]: {g_input}   (dim {len(g_input)})")
print("\nThe SAME z with a DIFFERENT label produces a different image:")
for k in [0, 3]:
    gi = np.concatenate([z, one_hot(k, n_classes)])
    print(f"  label {k} ('{classes[k]:<5}') -> input {gi}")

print("\nDiscriminator also receives C, so it must judge:")
print("  'is this a REAL image AND does it match the given label?'\n")

# ---- ACGAN ----
print("--- Auxiliary Classifier GAN (ACGAN) ---\n")
print("The discriminator has TWO outputs:")
print("  1. probability the input image is real")
print("  2. the class label of the input image (auxiliary classifier C)\n")

def softmax(v):
    e = np.exp(v - v.max()); return e / e.sum()

for tag, p_real, logits in [
    ("real 'bird' image     ", 0.93, np.array([0.2, 0.4, 3.1, 0.1, 0.3])),
    ("synthetic 'bird' image", 0.21, np.array([0.3, 0.5, 2.2, 0.2, 0.4])),
]:
    probs = softmax(logits)
    pred = int(np.argmax(probs))
    print(f"  {tag}: D_real = {p_real:.2f} | class probs = {np.round(probs,3)}")
    print(f"  {'':<22}  predicted class = '{classes[pred]}' (correct: bird)")

print("\nACGAN loss has two parts:")
print("  L_source = log P(real | x)     -> adversarial term")
print("  L_class  = log P(class | x)    -> classification term")
print("  D maximises L_source + L_class ; G maximises L_class - L_source")

print("\n--- Other conditional variants ---")
print("  BigGAN          : class-conditional GAN on ImageNet, built from ResBlocks")
print("  pix2pix (cGAN)  : C = image from the source domain (image-to-image)")
print("  Sketch-to-photo : C = a sketch")
print("  Text-to-image   : C = representation of a text prompt")

=== Conditional GAN (cGAN) ===

Both G and D are conditioned on additional information C.
C can be a class label, a source-domain image, a sketch, or text.

--- Class-Conditional GAN: C = class label ---

Random noise vector z (6-dim)      : [ 1.76  0.4   0.98  2.24  1.87 -0.98]
Class label            : 2 = 'bird'
One-hot encoding C     : [0. 0. 1. 0. 0.]
Generator input [z ; C]: [ 1.76  0.4   0.98  2.24  1.87 -0.98  0.    0.    1.    0.    0.  ]   (dim 11)

The SAME z with a DIFFERENT label produces a different image:
  label 0 ('cat  ') -> input [ 1.76  0.4   0.98  2.24  1.87 -0.98  1.    0.    0.    0.    0.  ]
  label 3 ('car  ') -> input [ 1.76  0.4   0.98  2.24  1.87 -0.98  0.    0.    0.    1.    0.  ]

Discriminator also receives C, so it must judge:
  'is this a REAL image AND does it match the given label?'

--- Auxiliary Classifier GAN (ACGAN) ---

The discriminator has TWO outputs:
  1. probability the input image is real
  2. the class label of the input image (auxiliary c

## Example 2: CycleGAN Cycle-Consistency Loss

This shows the idea that makes unpaired translation possible. An image is translated to the other domain and back; a well-trained pair of generators returns the original almost exactly (loss near 0), while poorly trained ones do not. The full objective combines both adversarial losses with the weighted cycle loss, and the SuperstarGAN FID table from the slide is reproduced at the end.

In [2]:
# Chapter 29 - Example 2: CycleGAN cycle-consistency loss
import numpy as np
np.random.seed(1)
np.set_printoptions(suppress=True)

print("=== CycleGAN: Unpaired Image-to-Image Translation ===\n")
print("pix2pix needs PAIRED data (same scene in both domains).")
print("CycleGAN works with UNPAIRED data using two generators and two")
print("discriminators, tied together by the cycle-consistency loss.\n")
print("  G_AB : domain A -> domain B      D_B : judges domain B images")
print("  G_BA : domain B -> domain A      D_A : judges domain A images\n")
print("Cycle consistency:  G_BA(G_AB(a)) ~ a    and    G_AB(G_BA(b)) ~ b\n")

# Represent tiny 3x3 'images'
a = np.round(np.random.rand(3, 3), 2)          # image in domain A (e.g. horse)
print("Original image a (domain A):"); print(a)

# Simulate two generators as invertible-ish transforms
def G_AB(x, quality=1.0):
    """A -> B (e.g. horse -> zebra). quality=1 means perfectly invertible."""
    noise = (1 - quality) * np.random.randn(*x.shape) * 0.3
    return np.clip(x * 0.8 + 0.15 + noise, 0, 1)

def G_BA(y, quality=1.0):
    """B -> A (inverse mapping)."""
    noise = (1 - quality) * np.random.randn(*y.shape) * 0.3
    return np.clip((y - 0.15) / 0.8 + noise, 0, 1)

print("\n--- Well-trained CycleGAN (quality = 1.0) ---")
b_hat = G_AB(a, 1.0)
a_cyc = G_BA(b_hat, 1.0)
print("Translated to domain B, G_AB(a):"); print(np.round(b_hat, 3))
print("Cycled back, G_BA(G_AB(a)):");      print(np.round(a_cyc, 3))
L_cyc_good = np.abs(a - a_cyc).mean()
print(f"Cycle-consistency loss  L1 = mean|a - G_BA(G_AB(a))| = {L_cyc_good:.6f}")

print("\n--- Poorly-trained CycleGAN (quality = 0.4) ---")
b_bad  = G_AB(a, 0.4)
a_bad  = G_BA(b_bad, 0.4)
print("Cycled back:"); print(np.round(a_bad, 3))
L_cyc_bad = np.abs(a - a_bad).mean()
print(f"Cycle-consistency loss  L1 = {L_cyc_bad:.6f}")

print(f"\nLoss comparison: good {L_cyc_good:.6f}  vs  bad {L_cyc_bad:.6f}")
print("The cycle loss forces the two generators to be near-inverses of each")
print("other, which is what prevents the network from ignoring the input and")
print("producing an arbitrary image in the target domain.\n")

# ---- Full CycleGAN objective ----
print("=== Full CycleGAN objective ===\n")
L_gan_AB, L_gan_BA, lam = 0.412, 0.389, 10.0
L_cycle = L_cyc_good + 0.0031
total = L_gan_AB + L_gan_BA + lam * L_cycle
print(f"  Adversarial loss  G_AB / D_B      : {L_gan_AB:.4f}")
print(f"  Adversarial loss  G_BA / D_A      : {L_gan_BA:.4f}")
print(f"  Cycle-consistency loss (both ways): {L_cycle:.4f}")
print(f"  lambda (cycle weight)             : {lam}")
print(f"  TOTAL = {L_gan_AB:.4f} + {L_gan_BA:.4f} + {lam}*{L_cycle:.4f} = {total:.4f}")

print("\n=== SuperstarGAN ===")
print("Combines ACGAN's separate classifier with CycleGAN's cycle loss.")
print("Reported FID on 10000 CelebA images (lower is better):")
for model, fid in [("SuperstarGAN", 22.7), ("StarGAN", 27.7),
                   ("SuperstarGAN without DA", 29.1),
                   ("StarGAN + SN + Hinge Loss", 34.0), ("StarGAN + DA", 38.3)]:
    print(f"  {model:<28} FID = {fid}")
print("  (DA = Data Augmentation, SN = Spectral Normalization)")

=== CycleGAN: Unpaired Image-to-Image Translation ===

pix2pix needs PAIRED data (same scene in both domains).
CycleGAN works with UNPAIRED data using two generators and two
discriminators, tied together by the cycle-consistency loss.

  G_AB : domain A -> domain B      D_B : judges domain B images
  G_BA : domain B -> domain A      D_A : judges domain A images

Cycle consistency:  G_BA(G_AB(a)) ~ a    and    G_AB(G_BA(b)) ~ b

Original image a (domain A):
[[0.42 0.72 0.  ]
 [0.3  0.15 0.09]
 [0.19 0.35 0.4 ]]

--- Well-trained CycleGAN (quality = 1.0) ---
Translated to domain B, G_AB(a):
[[0.486 0.726 0.15 ]
 [0.39  0.27  0.222]
 [0.302 0.43  0.47 ]]
Cycled back, G_BA(G_AB(a)):
[[0.42 0.72 0.  ]
 [0.3  0.15 0.09]
 [0.19 0.35 0.4 ]]
Cycle-consistency loss  L1 = mean|a - G_BA(G_AB(a))| = 0.000000

--- Poorly-trained CycleGAN (quality = 0.4) ---
Cycled back:
[[0.347 0.925 0.   ]
 [0.354 0.235 0.115]
 [0.    0.013 0.217]]
Cycle-consistency loss  L1 = 0.127930

Loss comparison: good 0.0000

## Example 3: Frechet Inception Distance from Scratch

This builds FID directly from its definition, comparing the mean and covariance of real and generated feature distributions. The experiments show FID near zero for a perfect match, rising sharply for a shifted mean, and - importantly - also rising when variety is too low, which is how FID detects the mode collapse that the Inception Score can miss.

In [3]:
# Chapter 29 - Example 3: Frechet Inception Distance (FID) from scratch
import numpy as np
from scipy.linalg import sqrtm
np.random.seed(2)

print("=== Frechet Inception Distance (FID) ===\n")
print("FID compares the DISTRIBUTIONS of Inception features for real and")
print("generated images, modelling each as a multivariate Gaussian:\n")
print("  FID = ||mu_r - mu_g||^2 + Tr( S_r + S_g - 2*(S_r S_g)^(1/2) )\n")
print("Lower FID = generated distribution closer to the real one.")
print("Unlike Inception Score, FID uses the REAL images as reference.\n")

def fid(f_real, f_gen):
    mu_r, mu_g = f_real.mean(axis=0), f_gen.mean(axis=0)
    S_r  = np.cov(f_real, rowvar=False)
    S_g  = np.cov(f_gen,  rowvar=False)
    diff = mu_r - mu_g
    covmean = sqrtm(S_r @ S_g)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(S_r + S_g - 2 * covmean))

n, dim = 500, 16
# Real image features
real = np.random.randn(n, dim)

experiments = {
  "Generated = real distribution (ideal)"   : np.random.randn(n, dim),
  "Slight mean shift (+0.3)"                : np.random.randn(n, dim) + 0.3,
  "Large mean shift (+1.5)"                 : np.random.randn(n, dim) + 1.5,
  "Correct mean, too little variety (0.4x)" : np.random.randn(n, dim) * 0.4,
  "Correct mean, too much variety (2.0x)"   : np.random.randn(n, dim) * 2.0,
}

print(f"Feature vectors: {n} images x {dim} dims (stand-in for Inception features)\n")
print(f"{'Generated-image distribution':<42} {'FID':>9}  {'Quality'}")
print("-" * 72)
for name, gen in experiments.items():
    score = fid(real, gen)
    q = ("excellent" if score < 1 else "good" if score < 5
         else "poor" if score < 20 else "very poor")
    print(f"  {name:<40} {score:>9.3f}  {q}")

print("\nObservations:")
print("  * Matching the real distribution gives FID near 0.")
print("  * A shifted mean (wrong content) raises FID sharply.")
print("  * Too little variety (mode collapse) ALSO raises FID, because the")
print("    covariance term no longer matches - FID penalises lost diversity.")

print("\n=== Inception Score vs FID ===\n")
print(f"  {'Aspect':<24} {'Inception Score':<26} {'FID'}")
print("  " + "-" * 70)
rows = [("Uses real images?",  "No",                    "Yes (reference set)"),
        ("Direction",          "Higher is better",      "Lower is better"),
        ("Detects mode collapse","Partially (marginal)", "Yes (covariance term)"),
        ("Based on",           "KL divergence",         "Frechet/Wasserstein-2 distance")]
for a, b, c in rows:
    print(f"  {a:<24} {b:<26} {c}")

=== Frechet Inception Distance (FID) ===

FID compares the DISTRIBUTIONS of Inception features for real and
generated images, modelling each as a multivariate Gaussian:

  FID = ||mu_r - mu_g||^2 + Tr( S_r + S_g - 2*(S_r S_g)^(1/2) )

Lower FID = generated distribution closer to the real one.
Unlike Inception Score, FID uses the REAL images as reference.

Feature vectors: 500 images x 16 dims (stand-in for Inception features)

Generated-image distribution                     FID  Quality
------------------------------------------------------------------------
  Generated = real distribution (ideal)        0.415  excellent
  Slight mean shift (+0.3)                     2.065  good
  Large mean shift (+1.5)                     36.850  very poor
  Correct mean, too little variety (0.4x)      6.011  poor
  Correct mean, too much variety (2.0x)       16.342  poor

Observations:
  * Matching the real distribution gives FID near 0.
  * A shifted mean (wrong content) raises FID sharply.
  * To